# PawCity AI — Resumable Model Comparisons

**IEOR 4570 · Project 2 · Deep Research path**

This notebook runs the three required same-model comparisons while respecting a conservative API request pace. It automatically saves results after each baseline and each completed agent answer. If an API rate limit or network interruption occurs, rerun the comparison cell to resume from the latest checkpoint.

**Important:** The notebook contains no secret keys. Do not save or share an API key in project files.


## 1. Locate the project directory

Open this notebook from the extracted project directory containing `app.py`, `pawcity/`, and `data/`.


In [ ]:
from pathlib import Path
import os
import sys
import json
import runpy

ROOT = Path.cwd()
if not (ROOT / "app.py").exists() and (ROOT / "PawCity_AI_Complete" / "app.py").exists():
    ROOT = ROOT / "PawCity_AI_Complete"
if not (ROOT / "app.py").exists():
    raise FileNotFoundError("Open this notebook inside the extracted PawCity AI project folder.")
sys.path.insert(0, str(ROOT))
print("Project directory:", ROOT.resolve())


## 2. Configure your API key securely

If your Jupyter kernel already has an `OPENAI_API_KEY`, this cell does not ask for it again. Otherwise, enter a new valid API key using the hidden password prompt. API usage is billed separately from a ChatGPT subscription.

The model is `gpt-4.1-mini` by default. To stay below the previously observed 10 requests/minute account limit, the pacing setting is **7 requests/minute**. Other processes or notebooks sharing the same account can still cause temporary 429 errors; the API client also retries those responses.


In [ ]:
from getpass import getpass

if not os.getenv("OPENAI_API_KEY"):
    os.environ["OPENAI_API_KEY"] = getpass("Enter your OpenAI API key: ")
os.environ.setdefault("PAWCITY_MODEL", "gpt-4.1-mini")
os.environ["PAWCITY_API_RPM"] = "7"

print("API key present:", bool(os.getenv("OPENAI_API_KEY")))
print("Model:", os.environ["PAWCITY_MODEL"])
print("Maximum requests/minute in this process:", os.environ["PAWCITY_API_RPM"])


## 3. Run or resume all three comparisons

**Real, billable model calls happen here.** Progress is saved to `reports/sample_comparisons.json`. On interruption, rerun this cell. Finished comparisons are skipped, and saved bare-model baseline responses are reused.

This notebook does not invent or recover earlier responses that were never saved by an older version of the script.


In [ ]:
assert os.getenv("OPENAI_API_KEY"), "The API key is not configured."
runpy.run_path(str(ROOT / "scripts" / "compare_samples.py"), run_name="__main__")


## 4. Review all genuine model outputs

When all three comparisons complete, this step shows a concise summary. The unedited full outputs are saved in `reports/sample_comparisons.json` and the README comparison table. Check the wording, citations and dates before submission.


In [ ]:
from IPython.display import display

report_path = ROOT / "reports" / "sample_comparisons.json"
if not report_path.exists():
    print("No comparison checkpoint has been created yet.")
else:
    data = json.loads(report_path.read_text(encoding="utf-8"))
    rows = [{
        "Question ID": r["id"],
        "Baseline saved": bool(r.get("same_model_no_system_no_tools_answer")),
        "Agent answer saved": bool(r.get("agent_answer")),
        "Tool calls": len(r.get("agent_tool_calls", [])),
        "Citations": len(r.get("agent_citations", [])),
    } for r in data.get("comparisons", [])]
    try:
        import pandas as pd
        display(pd.DataFrame(rows))
    except ImportError:
        for row in rows:
            print(row)
    print("Completed:", sum(r["Agent answer saved"] for r in rows), "/ 3")
